# Loan Approval Prediction System



In [2]:
from google.colab import files
uploaded=files.upload()

Saving Loan_default.csv to Loan_default.csv


In [3]:
import numpy as np
import pandas as pd
df=pd.read_csv("Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


# Business Problem
A bank wants to decide whether to approve a loan.


In [4]:
df.isnull().sum()

,0
LoanID,0
Age,0
Income,0
LoanAmount,0
CreditScore,0
MonthsEmployed,0
NumCreditLines,0
InterestRate,0
LoanTerm,0
DTIRatio,0


In [5]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

**Encoding**

In [6]:
label_encoder = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

**Seperate features and target**

In [7]:
X = df.drop("Default", axis=1)
y = df["Default"]

**Feature Scaling**


In [8]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Algorithms
**Logistic Regression**

Loan Approved / Rejected

**Random Forest**

Risk prediction

**K-Means**

Customer segmentation

**PCA**

Feature reduction


# Applying PCA

In [9]:
pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

print("Original Features:", X.shape[1])
print("Reduced Features:", X_pca.shape[1])

Original Features: 17
Reduced Features: 17


# Apply K-Means

In [10]:
kmeans = KMeans(n_clusters=3, random_state=42)

clusters = kmeans.fit_predict(X_pca)

df["Customer_Segment"] = clusters

print(df["Customer_Segment"].value_counts())

Customer_Segment
0    85212
1    85102
2    85033
Name: count, dtype: int64


In [11]:
X_pca_df = pd.DataFrame(X_pca)

X_pca_df["Segment"] = clusters

# SPlit dataset

In [12]:
X_pca_df.columns = X_pca_df.columns.astype(str)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_df,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
print(X_train.shape)
print(y_train.shape)

print(y.value_counts())

(204277, 18)
(204277,)
Default
0    225694
1     29653
Name: count, dtype: int64


In [15]:
print(X_train.columns)

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15', '16', 'Segment'],
      dtype='object')


# Train Random Forest

In [16]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

# Predict Risk

In [17]:
rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8853925983943607
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45170
           1       0.61      0.02      0.04      5900

    accuracy                           0.89     51070
   macro avg       0.75      0.51      0.49     51070
weighted avg       0.85      0.89      0.84     51070



# Train Logistic Regression

In [18]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

# Predict Approval

In [19]:
lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.8857450558057568
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45170
           1       0.61      0.03      0.06      5900

    accuracy                           0.89     51070
   macro avg       0.75      0.51      0.50     51070
weighted avg       0.86      0.89      0.84     51070



# PRedict for New Applicant

In [20]:

sample = X_test.iloc[[0]]

prediction = lr.predict(sample)

if prediction[0] == 0:
    print("Loan Approved")
else:
    print("Loan Rejected")

Loan Approved


# Save Models

In [21]:
import joblib

joblib.dump(rf, "random_forest.pkl")
joblib.dump(lr, "logistic_regression.pkl")
joblib.dump(pca, "pca.pkl")
joblib.dump(kmeans, "kmeans.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [22]:
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Random Forest Accuracy: 0.8853925983943607
Logistic Regression Accuracy: 0.8857450558057568


In [23]:
from google.colab import files

files.download("random_forest.pkl")
files.download("logistic_regression.pkl")
files.download("pca.pkl")
files.download("scaler.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
print(X.columns.tolist())

['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


In [25]:
files.download("kmeans.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
import os

for file in [
    "random_forest.pkl",
    "pca.pkl",
    "scaler.pkl",
    "kmeans.pkl"
]:
    print(file, round(os.path.getsize(file)/1024/1024,2), "MB")

random_forest.pkl 259.41 MB
pca.pkl 0.0 MB
scaler.pkl 0.0 MB
kmeans.pkl 0.98 MB


In [27]:
#retrianing the rf with smaller trees
from sklearn.ensemble import RandomForestClassifier

rf_small = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf_small.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=20,
                       random_state=42)

In [28]:
import joblib

joblib.dump(
    rf_small,
    "random_forest_small.pkl",
    compress=3
)

['random_forest_small.pkl']

In [29]:
import os

print(
    round(
        os.path.getsize("random_forest_small.pkl")/1024/1024,
        2
    ),
    "MB"
)

1.07 MB


In [30]:
files.download("random_forest_small.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>